# TimesFM-3: Zero-Shot Evaluation on TIME Benchmark (ICML 2026)

This notebook evaluates **TimesFM-3** on the complete **TIME Benchmark** (50 fresh datasets spanning Nature, Energy, Transportation, Healthcare, Finance, Economics, Manufacturing, and Cloud domains) across all horizon regimes (`short`, `medium`, `long`).

### Evaluation Configuration:
- **Model Name**: `TimesFM-3`
- **Architecture**: TimesFM-3 PyTorch with cross-variate attention (`use_variate_attention=True`)
- **Mode**: Full native multivariate (`to_univariate=False`), matching the Chronos-2 evaluation setup
- **Variate Chunking**: Handled automatically by `TimesFM3Evaluator` for datasets with $> 32$ variates
- **Batch Sizing**: Adaptive batch sizing scaled by token volume ($64 / (\min(32, V) \cdot L_{\text{norm}})$) rounded to nearest power of 2, exactly as in `fev_bench`
- **Output Results**: Saved to `TimesFM-3_multivariate/time/`


In [ ]:
import os
import sys
import math
import json
import warnings
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from gluonts.time_feature import get_seasonality
from huggingface_hub import snapshot_download

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def find_repo_root():
    curr = Path.cwd().resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'src' / 'timesfm3').exists() or (p / 'pyproject.toml').exists():
            return p
    return curr

repo_root = find_repo_root()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


from timesfm3 import TimesFM3Evaluator, ModelConfig
from timebench.evaluation.data import Dataset, load_dataset_config, get_dataset_settings
from timebench.evaluation.saver import save_window_predictions
from timebench.evaluation.utils import get_available_terms

print(f"PyTorch version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}", flush=True)


In [ ]:
# Ensure TIME datasets are available in cache
time_cache_dir = Path.home() / ".cache" / "TIME_dataset"
time_cache_dir.mkdir(parents=True, exist_ok=True)

if not any(time_cache_dir.iterdir()):
    print(f"Downloading Real-TSF/TIME datasets to {time_cache_dir}...", flush=True)
    snapshot_download(
        repo_id="Real-TSF/TIME",
        repo_type="dataset",
        local_dir=str(time_cache_dir),
    )
    print("Download completed!", flush=True)
else:
    print(f"TIME dataset cache verified at: {time_cache_dir}", flush=True)

# Load benchmark configuration
config_path = repo_root / "TIME/src/timebench/config/datasets.yaml"
bench_config = load_dataset_config(config_path)
all_datasets = list(bench_config.get("datasets", {}).keys())
print(f"Loaded {len(all_datasets)} total dataset configurations from datasets.yaml", flush=True)

# Output directory destination
output_dir = repo_root / "TimesFM-3_multivariate/time"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved to: {output_dir}", flush=True)


In [ ]:
# Initialize TimesFM-3 Evaluator
forecaster = TimesFM3Evaluator(ModelConfig(
    checkpoint_path=os.getenv("TIMESFM3_CHECKPOINT", "google/timesfm-3.0-pytorch"),
    per_core_batch_size=64,
    use_variate_attention=True,
    use_sdpa=True,
))
print(f"TimesFM-3 Evaluator loaded on: {forecaster.device}", flush=True)

def get_optimal_batch_size(
    max_context_len: int,
    num_variates: int = 1,
    min_batch: int = 4,
    max_batch: int = 64,
) -> int:
    """Compute dynamic batch size scaled by total variates (capped at 32 chunk limit) and context length, rounded to nearest power of 2."""
    full_context_count_per_ts = min(32, num_variates) * min(1.0, max(max_context_len, 32) / 15360.0)
    batch_size = 64.0 / max(full_context_count_per_ts, 0.01)
    power_of_2 = int(2 ** round(math.log2(batch_size))) if batch_size > 0 else 1
    return int(np.clip(power_of_2, min_batch, max_batch))

def prepare_time_context(item: dict, max_context_length: int = 15360) -> np.ndarray:
    """Extract target array from TIME dataset entry and format as (num_variates, context_len)."""
    target = np.asarray(item["target"], dtype=np.float32)
    if target.ndim == 1:
        target = target[np.newaxis, :]
    if target.shape[-1] > max_context_length:
        target = target[:, -max_context_length:]
    return target


In [ ]:
# Execute Evaluation Across all 50 TIME Datasets
quantile_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
max_context_length = 15360
min_batch = 4
max_batch = 64

total_datasets = len(all_datasets)

for ds_idx, dataset_name in enumerate(all_datasets, 1):
    terms = get_available_terms(dataset_name, bench_config)
    if not terms:
        continue

    print(f"\n{'='*75}", flush=True)
    print(f"[{ds_idx}/{total_datasets}] Dataset: {dataset_name} | Available Terms: {terms}", flush=True)
    print(f"{'='*75}", flush=True)

    for term in terms:
        ds_config = f"{dataset_name}/{term}"
        target_metrics_path = output_dir / ds_config / "metrics.npz"

        if target_metrics_path.exists() and target_metrics_path.stat().st_size > 0:
            print(f"  -> Term '{term}': Skipping (already evaluated at {target_metrics_path})", flush=True)
            continue

        try:
            settings = get_dataset_settings(dataset_name, term, bench_config)
            prediction_length = settings.get("prediction_length")
            test_length = settings.get("test_length")
            val_length = settings.get("val_length", 0)

            dataset = Dataset(
                name=dataset_name,
                term=term,
                to_univariate=False, # Full native multivariate
                prediction_length=prediction_length,
                test_length=test_length,
                val_length=val_length,
                storage_path=time_cache_dir,
            )

            eval_data = dataset.test_data
            eval_input_list = list(eval_data.input)
            total_items = len(eval_input_list)
            num_variates = dataset.target_dim
            season_length = get_seasonality(dataset.freq)

            # Calculate optimal dynamic batch size matching fev_bench
            max_ctx_in_task = max((np.asarray(inp["target"]).shape[-1] for inp in eval_input_list), default=32)
            dynamic_batch_size = get_optimal_batch_size(
                max_context_len=max_ctx_in_task,
                num_variates=num_variates,
                min_batch=min_batch,
                max_batch=max_batch,
            )
            total_batches = math.ceil(total_items / dynamic_batch_size)

            print(f"  -> Term '{term}': Horizon={prediction_length}, Windows={dataset.windows}, Total Items={total_items}, Variates={num_variates} | Batch Size={dynamic_batch_size}, Total Batches={total_batches}", flush=True)

            fc_quantiles_batches = []

            for b_start in range(0, total_items, dynamic_batch_size):
                b_end = min(b_start + dynamic_batch_size, total_items)
                batch_contexts = [prepare_time_context(eval_input_list[i], max_context_length=max_context_length) for i in range(b_start, b_end)]

                batch_outs = list(forecaster.predict_batch(
                    contexts=batch_contexts,
                    horizon=prediction_length,
                    return_quantiles=True,
                    use_symmetric_averaging=True,
                    make_positive=True,
                    sort_quantiles=True,
                ))

                batch_q_list = []
                for out in batch_outs:
                    # out.quantiles shape: (num_variates, prediction_length, num_quantiles)
                    # Permute to expected TIME shape: (num_quantiles, num_variates, prediction_length)
                    q = out.quantiles.transpose(2, 0, 1)
                    batch_q_list.append(q[np.newaxis, ...])

                batch_q_arr = np.concatenate(batch_q_list, axis=0)
                fc_quantiles_batches.append(batch_q_arr)

                current_batch_idx = (b_start // dynamic_batch_size) + 1
                print(f"    Batch [{current_batch_idx}/{total_batches}] ({len(batch_contexts)} series) completed", flush=True)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            fc_quantiles = np.concatenate(fc_quantiles_batches, axis=0)

            model_hyperparams = {
                "model": "TimesFM-3",
                "context_length": max_context_length,
                "quantile_levels": quantile_levels,
                "use_variate_attention": True,
            }

            metadata = save_window_predictions(
                dataset=dataset,
                fc_quantiles=fc_quantiles,
                ds_config=ds_config,
                output_base_dir=str(output_dir),
                seasonality=season_length,
                model_hyperparams=model_hyperparams,
                quantile_levels=quantile_levels,
            )
            print(f"  [Saved] {ds_config} -> {output_dir / ds_config}", flush=True)

        except Exception as e:
            print(f"  ERROR on {ds_config}: {e}", flush=True)

print(f"\nAll TIME Benchmark evaluations completed! Results saved to: {output_dir}", flush=True)


In [ ]:
# Aggregation and Leaderboard Computation
print("=" * 75, flush=True)
print("COMPUTING TIME BENCHMARK SUMMARY & LEADERBOARD METRICS", flush=True)
print("=" * 75, flush=True)

results_list = []

for dataset_name in all_datasets:
    terms = get_available_terms(dataset_name, bench_config)
    for term in terms:
        ds_config = f"{dataset_name}/{term}"
        metrics_path = output_dir / ds_config / "metrics.npz"
        config_json_path = output_dir / ds_config / "config.json"

        if metrics_path.exists() and config_json_path.exists():
            try:
                metrics = np.load(metrics_path)
                with open(config_json_path, "r") as f:
                    cfg = json.load(f)

                row = {
                    "dataset": dataset_name,
                    "term": term,
                    "config": ds_config,
                    "MASE": float(np.nanmean(metrics["mase"])),
                    "CRPS": float(np.nanmean(metrics["crps"])),
                    "WAPE": float(np.nanmean(metrics.get("wape", np.nan))),
                    "MSE": float(np.nanmean(metrics["mse"])),
                    "MAE": float(np.nanmean(metrics["mae"])),
                    "RMSE": float(np.nanmean(metrics["rmse"])),
                    "num_series": cfg.get("num_series", 1),
                    "windows": cfg.get("windows", 1),
                    "prediction_length": cfg.get("prediction_length", 1),
                }
                results_list.append(row)
            except Exception as e:
                pass

if results_list:
    summary_df = pd.DataFrame(results_list)
    csv_summary_path = output_dir / "timesfm3_time_bench_summary.csv"
    summary_df.to_csv(csv_summary_path, index=False)
    print(f"\nEvaluated Tasks: {len(summary_df)}", flush=True)
    print(f"Overall Average MASE: {summary_df['MASE'].mean():.4f}", flush=True)
    print(f"Overall Average CRPS: {summary_df['CRPS'].mean():.4f}", flush=True)
    print(f"Overall Average WAPE: {summary_df['WAPE'].mean():.4f}", flush=True)
    print(f"Summary CSV saved to: {csv_summary_path}", flush=True)
    print("\n--- First 15 Task Results ---", flush=True)
    print(summary_df[["dataset", "term", "MASE", "CRPS", "WAPE", "RMSE"]].head(15).to_string(index=False), flush=True)
else:
    print("No evaluated task metrics found yet.", flush=True)
